# Assessment 2 - Financial Accounting & GL Reconciliation

Finance reports that balances generated from the new data platform do not reconcile with the
bank's General Ledger. This notebook traces the cause across four tasks: GL arithmetic integrity
and reconciliation, accounting mapping validation, a structured variance investigation, and a
reconciliation framework design. See `results/assessment-2/assessment-2-overview.md` for the full
scenario, table shapes, and scale. Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - Validate Accounting Integrity

Confirm `opening_balance + debit_movement - credit_movement = closing_balance` on the General
Ledger, identify violations, then independently recompute expected debit/credit movements from the
transaction-level data and reconcile against the General Ledger at legal entity, GL account, cost
center, currency, and accounting date.


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task1-gl-integrity")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


gl_df = jdbc_table("finance.gl_balance")
txn_df = jdbc_table("bronze.finance_transactions")

print(f"[INFO] finance.gl_balance row_count={gl_df.count()}")
print(f"[INFO] bronze.finance_transactions row_count={txn_df.count()}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/09 11:32:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[INFO] finance.gl_balance row_count=589


[INFO] bronze.finance_transactions row_count=1523


### Arithmetic integrity

`opening_balance + debit_movement - credit_movement = closing_balance` checked on every General
Ledger row. Tolerance: exact equality - all four columns are `decimal(20,2)` and the expression is
pure addition/subtraction, so a nonzero result is a genuine arithmetic break, not a rounding artifact.


In [3]:
from pyspark.sql.functions import col, sum as spark_sum, abs as spark_abs

# kept as native decimal(20,2) here (no cast to double) - exact-equality comparisons need to stay in
# decimal arithmetic; a double cast introduces float rounding noise well below the cent level, which
# would silently inflate this count. Double is used further down only where a numeric tolerance
# (e.g. one cent) already absorbs that noise.
gl_exact = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("opening_balance"), col("debit_movement"), col("credit_movement"), col("closing_balance"),
)

arithmetic_check = gl_exact.withColumn(
    "computed_closing", col("opening_balance") + col("debit_movement") - col("credit_movement")
).withColumn(
    "variance", col("closing_balance") - col("computed_closing")
)

arithmetic_violations = arithmetic_check.filter(col("variance") != 0)
violation_count = arithmetic_violations.count()

print(f"[INFO] arithmetic integrity violations={violation_count}")
arithmetic_violations.select(
    "accounting_date", "legal_entity", "gl_account", "cost_center", "currency",
    "closing_balance", "computed_closing", "variance",
).show(20, truncate=False)


[INFO] arithmetic integrity violations=5


+---------------+------------+----------+-----------+--------+---------------+----------------+--------+
|accounting_date|legal_entity|gl_account|cost_center|currency|closing_balance|computed_closing|variance|
+---------------+------------+----------+-----------+--------+---------------+----------------+--------+
|2026-08-17     |LE3         |GL1014    |CC05       |SGD     |37363.87       |39698.69        |-2334.82|
|2026-08-17     |LE4         |GL1007    |CC08       |USD     |-11176.81      |-9612.03        |-1564.78|
|2026-08-20     |LE1         |GL1003    |CC04       |SGD     |82746.17       |79671.14        |3075.03 |
|2026-08-21     |LE4         |GL1009    |CC10       |USD     |85752.25       |83837.66        |1914.59 |
|2026-08-21     |LE4         |GL1013    |CC04       |SGD     |169471.57      |172377.24       |-2905.67|
+---------------+------------+----------+-----------+--------+---------------+----------------+--------+



### Independent movement recomputation

The General Ledger's five-dimension grouping key (`accounting_date, legal_entity, gl_account,
cost_center, currency`) is also the grouping key this recomputation aggregates the transaction-level
data onto, joining `posting_date` to `accounting_date` - the Ledger is dated by *posting*, not by
`transaction_date`.

Tolerance: `0.01` (one minor-currency-unit) applied independently to each side of the movement -
loose enough to absorb rounding noise already present in the amounts, tight enough that it never
masks a genuine one-record miss. A full outer join (not left/inner) so a Ledger key with no matching
transactions, or a transaction key with no matching Ledger row, both surface as a variance instead of
silently dropping out of the comparison.


In [4]:
from pyspark.sql.functions import when

MOVEMENT_TOLERANCE_ABS = 0.01

# double is safe here - comparisons below use a 0.01 tolerance, well above any float rounding noise.
gl = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("debit_movement").cast("double").alias("debit_movement"),
    col("credit_movement").cast("double").alias("credit_movement"),
)

txn = txn_df.select(
    col("posting_date").alias("accounting_date"),
    col("legal_entity"), col("gl_account"), col("cost_center"), col("currency"),
    col("debit_credit_indicator"),
    col("local_amount").cast("double").alias("local_amount"),
)

recomputed = txn.groupBy("accounting_date", "legal_entity", "gl_account", "cost_center", "currency").agg(
    spark_sum(when(col("debit_credit_indicator") == "DEBIT", col("local_amount")).otherwise(0.0)).alias("recomputed_debit"),
    spark_sum(when(col("debit_credit_indicator") == "CREDIT", col("local_amount")).otherwise(0.0)).alias("recomputed_credit"),
)

JOIN_KEYS = ["accounting_date", "legal_entity", "gl_account", "cost_center", "currency"]

movement_compare = gl.join(recomputed, JOIN_KEYS, "full_outer").select(
    *[col(k) for k in JOIN_KEYS],
    col("debit_movement"), col("recomputed_debit"),
    col("credit_movement"), col("recomputed_credit"),
).fillna(0.0, subset=["debit_movement", "recomputed_debit", "credit_movement", "recomputed_credit"]).withColumn(
    "debit_variance", col("debit_movement") - col("recomputed_debit")
).withColumn(
    "credit_variance", col("credit_movement") - col("recomputed_credit")
)

movement_variances = movement_compare.filter(
    (spark_abs(col("debit_variance")) > MOVEMENT_TOLERANCE_ABS)
    | (spark_abs(col("credit_variance")) > MOVEMENT_TOLERANCE_ABS)
)
movement_compare.cache()
movement_variance_count = movement_variances.count()

print(f"[INFO] movement recomputation variances={movement_variance_count}")
movement_variances.orderBy(spark_abs(col("debit_variance") + col("credit_variance")).desc()).show(20, truncate=False)


[INFO] movement recomputation variances=284


+---------------+------------+----------+-----------+--------+--------------+------------------+---------------+------------------+-------------------+-------------------+
|accounting_date|legal_entity|gl_account|cost_center|currency|debit_movement|recomputed_debit  |credit_movement|recomputed_credit |debit_variance     |credit_variance    |
+---------------+------------+----------+-----------+--------+--------------+------------------+---------------+------------------+-------------------+-------------------+
|2026-08-19     |LE2         |GL9999    |CC99       |EUR     |39440.74      |57310.36          |22662.42       |32930.85          |-17869.620000000003|-10268.43          |
|2026-08-20     |LE3         |GL9999    |CC99       |USD     |26677.86      |35880.07          |50026.3        |66845.22          |-9202.21           |-16818.92          |
|2026-08-19     |LE4         |GL9999    |CC99       |USD     |868.14        |1156.61           |72566.0        |96737.51          |-288.4699

### Dimensional reconciliation

The same recomputation, rolled up to one dimension at a time instead of the full five-key grain.
Reconciliation status per row: `PASS` if `variance_pct < 0.1%`, `WARNING` if `< 1%`, `FAIL` otherwise.


In [5]:
def status_for(variance_pct):
    pct = abs(variance_pct)
    if pct < 0.1:
        return "PASS"
    if pct < 1.0:
        return "WARNING"
    return "FAIL"


LEVEL_DIMENSIONS = ["legal_entity", "gl_account", "cost_center", "currency", "accounting_date"]

dimension_summaries = {}

for dim in LEVEL_DIMENSIONS:
    rolled = movement_compare.groupBy(dim).agg(
        spark_sum("debit_movement").alias("gl_debit"),
        spark_sum("recomputed_debit").alias("recomputed_debit"),
        spark_sum("credit_movement").alias("gl_credit"),
        spark_sum("recomputed_credit").alias("recomputed_credit"),
    ).withColumn(
        "debit_variance", col("gl_debit") - col("recomputed_debit")
    ).withColumn(
        "credit_variance", col("gl_credit") - col("recomputed_credit")
    ).collect()

    rows = []
    for r in rolled:
        gl_total = (r["gl_debit"] or 0.0) + (r["gl_credit"] or 0.0)
        var_total = (r["debit_variance"] or 0.0) + (r["credit_variance"] or 0.0)
        variance_pct = round((abs(var_total) / gl_total * 100) if gl_total else 0.0, 4)
        rows.append({
            "group_value": r[dim], "gl_debit": r["gl_debit"], "gl_credit": r["gl_credit"],
            "debit_variance": r["debit_variance"], "credit_variance": r["credit_variance"],
            "variance_pct": variance_pct, "status": status_for(variance_pct),
        })
    dimension_summaries[dim] = rows
    worst = sorted(rows, key=lambda r: abs(r["variance_pct"]), reverse=True)[:5]
    print(f"[INFO] dimensional reconciliation by {dim} - top variance groups:")
    for r in worst:
        print(f"  [{r['status']}] {dim}={r['group_value']} debit_variance={r['debit_variance']:.2f} "
              f"credit_variance={r['credit_variance']:.2f} variance_pct={r['variance_pct']}%")


[INFO] dimensional reconciliation by legal_entity - top variance groups:
  [FAIL] legal_entity=LE4 debit_variance=-230710.45 credit_variance=-214277.49 variance_pct=12.2325%
  [FAIL] legal_entity=LE3 debit_variance=-202704.38 credit_variance=-255233.01 variance_pct=11.4772%
  [FAIL] legal_entity=LE2 debit_variance=-209941.98 credit_variance=-173487.86 variance_pct=11.3555%
  [FAIL] legal_entity=LE1 debit_variance=-205855.71 credit_variance=-199604.04 variance_pct=9.4227%


[INFO] dimensional reconciliation by gl_account - top variance groups:
  [FAIL] gl_account=GL1009 debit_variance=-116933.09 credit_variance=0.00 variance_pct=14.3507%
  [FAIL] gl_account=GL1004 debit_variance=-113084.97 credit_variance=0.00 variance_pct=13.2607%
  [FAIL] gl_account=GL1007 debit_variance=0.00 credit_variance=-113767.37 variance_pct=12.5202%
  [FAIL] gl_account=GL1013 debit_variance=-92280.61 credit_variance=0.00 variance_pct=11.8572%
  [FAIL] gl_account=GL1011 debit_variance=-98341.46 credit_variance=0.00 variance_pct=11.7118%


[INFO] dimensional reconciliation by cost_center - top variance groups:
  [FAIL] cost_center=CC10 debit_variance=-116933.09 credit_variance=0.00 variance_pct=14.3507%
  [FAIL] cost_center=CC08 debit_variance=0.00 credit_variance=-111961.34 variance_pct=12.6558%
  [FAIL] cost_center=CC05 debit_variance=-182485.01 credit_variance=-2712.39 variance_pct=11.4337%
  [FAIL] cost_center=CC99 debit_variance=-90289.96 credit_variance=-340405.49 variance_pct=11.0748%
  [FAIL] cost_center=CC01 debit_variance=-75094.56 credit_variance=-96062.97 variance_pct=10.8393%


[INFO] dimensional reconciliation by currency - top variance groups:
  [FAIL] currency=EUR debit_variance=-325397.67 credit_variance=-331201.88 variance_pct=45.7804%
  [FAIL] currency=USD debit_variance=-523814.85 credit_variance=-511400.52 variance_pct=34.1154%
  [PASS] currency=SGD debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%


[INFO] dimensional reconciliation by accounting_date - top variance groups:
  [FAIL] accounting_date=2026-08-19 debit_variance=-213002.57 credit_variance=-151971.43 variance_pct=11.8745%
  [FAIL] accounting_date=2026-08-21 debit_variance=-148435.87 credit_variance=-213536.53 variance_pct=11.6854%
  [FAIL] accounting_date=2026-08-18 debit_variance=-169638.78 credit_variance=-144322.85 variance_pct=10.7357%
  [FAIL] accounting_date=2026-08-20 debit_variance=-173426.88 credit_variance=-154774.23 variance_pct=10.4985%
  [FAIL] accounting_date=2026-08-17 debit_variance=-144708.42 credit_variance=-177997.36 variance_pct=10.4588%


### Reconciliation framework write-back

The reconciliation control table's `dimension` column is a closed set (`row_count`, `amount`); the
fine-grained per-dimension detail above is reported in this notebook and the reconciliation-results
deliverable only. `row_count` compares total transaction count against total Ledger row count;
`amount` compares total transaction value against total Ledger closing-balance value.


In [6]:
def reserve_batch_id(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT nextval('reconciliation.rc_batch_control_batch_id_seq');")
        return cur.fetchone()[0]


def insert_batch_control(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_batch_control (batch_id, batch_date, assessment_id, status) "
            "VALUES (%s, CURRENT_DATE, %s, %s)",
            (batch_id, "assessment-2", status),
        )


def update_batch_status(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "UPDATE reconciliation.rc_batch_control SET status = %s WHERE batch_id = %s",
            (status, batch_id),
        )


def insert_result_row(conn, batch_id, dimension, source_value, target_value):
    variance = target_value - source_value
    variance_pct = round((variance / source_value * 100) if source_value else 0.0, 4)
    status = status_for(variance_pct)
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_reconciliation_results "
            "(batch_id, dimension, source_value, target_value, variance, variance_pct, reconciliation_status) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            (batch_id, dimension, source_value, target_value, variance, variance_pct, status),
        )
    return status


txn_count = txn_df.count()
gl_count = gl_df.count()
txn_amount = txn_df.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
gl_amount = gl_df.agg(spark_sum(col("closing_balance").cast("double"))).collect()[0][0] or 0.0

conn = psycopg2.connect(host="postgres", port=5432, dbname=POSTGRES_DB, user=POSTGRES_USER, password=POSTGRES_PASSWORD)
batch_id = reserve_batch_id(conn)
insert_batch_control(conn, batch_id, "RUNNING")

statuses = [
    insert_result_row(conn, batch_id, "row_count", txn_count, gl_count),
    insert_result_row(conn, batch_id, "amount", txn_amount, gl_amount),
]
overall_status = max(statuses, key=lambda s: {"PASS": 0, "WARNING": 1, "FAIL": 2}[s])
update_batch_status(conn, batch_id, overall_status)
conn.commit()
conn.close()

print(f"[INFO] source(transactions): count={txn_count} amount={txn_amount:.2f}")
print(f"[INFO] target(general ledger): count={gl_count} amount={gl_amount:.2f}")
print(f"[{overall_status}] task 1 batch reconciliation: batch_id={batch_id}")


[INFO] source(transactions): count=1523 amount=16999151.01
[INFO] target(general ledger): count=589 amount=12996847.89
[FAIL] task 1 batch reconciliation: batch_id=17


In [7]:
spark.stop()


## Task 2 - Validate Accounting Mapping

Using the accounting mapping table: determine whether transactions are posted to the expected GL
accounts, validate mapping effective dates, identify transactions with missing accounting mappings,
detect overlapping effective-date mappings, identify expired mappings still being used, and identify
products mapped to multiple GL accounts unexpectedly.

Implemented below via Spark SQL over temp views rather than the DataFrame API - the effective-dated
join and the self-joins for overlap/multi-GL detection read directly as SQL.


In [8]:
from pyspark.sql.functions import lit

spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task2-mapping-validation")
    .getOrCreate()
)

txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")

txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")

print(f"[INFO] bronze.finance_transactions row_count={txn_df.count()}")
print(f"[INFO] ref.accounting_mapping row_count={map_df.count()}")


[INFO] bronze.finance_transactions row_count=1523


[INFO] ref.accounting_mapping row_count=22


### Join key

The mapping table's transaction type and the transaction table's debit/credit indicator carry the
same domain (`DEBIT`/`CREDIT`) under different column names; every check below effective-dates the
join on the transaction's own `transaction_date` (not `posting_date` - the mapping rule governs which
policy applied when the transaction occurred, independent of when it was later posted).


### Posted to expected GL account

If the join returns more than one mapping row per transaction (an overlapping-range case), every
matched row is evaluated independently rather than one being picked arbitrarily - a transaction is
flagged if it disagrees with *any* matched mapping.


In [9]:
gl_mismatch = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           m.expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON  t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
      AND t.transaction_date >= m.effective_start_date
      AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    WHERE t.gl_account <> m.expected_gl_account
''').withColumn("exception", lit("GL_MISMATCH"))

gl_mismatch_rows = gl_mismatch.count()
gl_mismatch_txns = gl_mismatch.select("transaction_id").distinct().count()
print(f"[INFO] posted to expected GL account - GL_MISMATCH rows={gl_mismatch_rows} distinct_transactions={gl_mismatch_txns}")
gl_mismatch.show(10, truncate=False)


[INFO] posted to expected GL account - GL_MISMATCH rows=403 distinct_transactions=319


+--------------+------------+---------+-------------------+---------------+-----------+
|transaction_id|product_code|actual_gl|expected_gl_account|accounting_date|exception  |
+--------------+------------+---------+-------------------+---------------+-----------+
|FTX-0000008   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000012   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000020   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000023   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000027   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000042   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000080   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000100   |P1          |GL1007   |GL1014             |2026-08-17     |GL_MISMATCH|
|FTX-0000114   |P1          |GL1

### Mapping effective-date validity and missing mapping

A mapping-effective-date violation is a `(product_code, debit_credit_indicator)` pair that exists in
the mapping table but for which no row's window covers `transaction_date`. A missing mapping is
distinguished from that by whether *any* row exists for the pair at all, not just whether one covers
the right date.


In [10]:
no_effective_mapping = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           CAST(NULL AS STRING) AS expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    WHERE EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
    )
    AND NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
''').withColumn("exception", lit("NO_EFFECTIVE_MAPPING"))

mapping_not_found = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           CAST(NULL AS STRING) AS expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
    )
''').withColumn("exception", lit("MAPPING_NOT_FOUND"))

print(f"[INFO] mapping effective-date validity - NO_EFFECTIVE_MAPPING rows={no_effective_mapping.count()}")
print(f"[INFO] missing accounting mapping - MAPPING_NOT_FOUND rows={mapping_not_found.count()}")
mapping_not_found.show(5, truncate=False)


[INFO] mapping effective-date validity - NO_EFFECTIVE_MAPPING rows=0


[INFO] missing accounting mapping - MAPPING_NOT_FOUND rows=385


+--------------+------------+---------+-------------------+---------------+-----------------+
|transaction_id|product_code|actual_gl|expected_gl_account|accounting_date|exception        |
+--------------+------------+---------+-------------------+---------------+-----------------+
|FTX-0000019   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000026   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000028   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000043   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
|FTX-0000051   |P10         |GL9999   |NULL               |2026-08-17     |MAPPING_NOT_FOUND|
+--------------+------------+---------+-------------------+---------------+-----------------+
only showing top 5 rows



### Overlapping effective-date mapping ranges

Mapping-level, not transaction-level: two rows for the same `(product_code, transaction_type)` whose
windows intersect.


In [11]:
overlapping_mapping = spark.sql('''
    SELECT a.product_code, a.transaction_type, a.effective_start_date, a.effective_end_date,
           b.effective_start_date AS overlap_start, b.effective_end_date AS overlap_end
    FROM accounting_mapping a
    JOIN accounting_mapping b
      ON  a.product_code = b.product_code AND a.transaction_type = b.transaction_type
      AND a.effective_start_date < b.effective_start_date
      AND a.effective_start_date <= COALESCE(b.effective_end_date, DATE '9999-12-31')
      AND COALESCE(a.effective_end_date, DATE '9999-12-31') >= b.effective_start_date
''')

overlapping_mapping_count = overlapping_mapping.count()
print(f"[INFO] overlapping effective-date mapping ranges - pairs={overlapping_mapping_count}")
overlapping_mapping.show(10, truncate=False)


[INFO] overlapping effective-date mapping ranges - pairs=8


+------------+----------------+--------------------+------------------+-------------+-----------+
|product_code|transaction_type|effective_start_date|effective_end_date|overlap_start|overlap_end|
+------------+----------------+--------------------+------------------+-------------+-----------+
|P1          |CREDIT          |2025-08-17          |NULL              |2026-07-18   |NULL       |
|P1          |DEBIT           |2025-08-17          |NULL              |2026-07-18   |NULL       |
|P1          |DEBIT           |2025-08-17          |NULL              |2026-08-12   |NULL       |
|P1          |DEBIT           |2026-07-18          |NULL              |2026-08-12   |NULL       |
|P4          |DEBIT           |2025-08-17          |NULL              |2026-01-29   |2026-08-07 |
|P5          |DEBIT           |2025-08-17          |NULL              |2026-08-12   |NULL       |
|P8          |CREDIT          |2025-08-17          |NULL              |2026-01-29   |2026-08-07 |
|P9          |CREDIT

### Expired mapping still referenced

A transaction can reference an expired mapping row while *also* having a currently-valid mapping row
it should have used instead - that combination is a GL mismatch, not an expired-mapping case. This
fires only when the expired row is the sole candidate.


In [12]:
expired_mapping = spark.sql('''
    SELECT t.transaction_id, t.product_code, t.gl_account AS actual_gl,
           m.expected_gl_account, t.posting_date AS accounting_date
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
    WHERE m.effective_end_date IS NOT NULL
      AND t.transaction_date > m.effective_end_date
      AND NOT EXISTS (
        SELECT 1 FROM accounting_mapping m2
        WHERE m2.product_code = t.product_code AND m2.transaction_type = t.debit_credit_indicator
          AND t.transaction_date >= m2.effective_start_date
          AND (t.transaction_date <= m2.effective_end_date OR m2.effective_end_date IS NULL)
      )
''').withColumn("exception", lit("EXPIRED_MAPPING"))

expired_mapping_rows = expired_mapping.count()
print(f"[INFO] expired mapping still referenced rows={expired_mapping_rows}")
expired_mapping.show(10, truncate=False)


[INFO] expired mapping still referenced rows=0


+--------------+------------+---------+-------------------+---------------+---------+
|transaction_id|product_code|actual_gl|expected_gl_account|accounting_date|exception|
+--------------+------------+---------+-------------------+---------------+---------+
+--------------+------------+---------+-------------------+---------------+---------+



### Product mapped to multiple GL accounts unexpectedly

Mapping-level: currently-active rows (open-ended or not yet expired) that disagree on the expected GL
account - a genuine data conflict rather than a time-ordered supersession.


In [13]:
multi_gl_mapping = spark.sql('''
    SELECT product_code, transaction_type, COUNT(DISTINCT expected_gl_account) AS gl_account_count
    FROM accounting_mapping
    WHERE effective_end_date IS NULL OR effective_end_date >= CURRENT_DATE
    GROUP BY product_code, transaction_type
    HAVING COUNT(DISTINCT expected_gl_account) > 1
''')

multi_gl_mapping_count = multi_gl_mapping.count()
print(f"[INFO] product mapped to multiple GL accounts - product/type pairs={multi_gl_mapping_count}")
multi_gl_mapping.show(10, truncate=False)


[INFO] product mapped to multiple GL accounts - product/type pairs=4


+------------+----------------+----------------+
|product_code|transaction_type|gl_account_count|
+------------+----------------+----------------+
|P1          |DEBIT           |3               |
|P9          |CREDIT          |2               |
|P1          |CREDIT          |2               |
|P5          |DEBIT           |2               |
+------------+----------------+----------------+



### Exception output

The assignment's own shape - `Transaction, Product, Actual GL, Expected GL, Accounting Date,
Exception` - applies to the four per-transaction checks above; the two mapping-level findings (no
`transaction_id` to key on) are reported separately.


In [14]:
mapping_exceptions = (
    gl_mismatch
    .unionByName(no_effective_mapping)
    .unionByName(mapping_not_found)
    .unionByName(expired_mapping)
)

exception_output = mapping_exceptions.select(
    col("transaction_id").alias("Transaction"),
    col("product_code").alias("Product"),
    col("actual_gl").alias("Actual GL"),
    col("expected_gl_account").alias("Expected GL"),
    col("accounting_date").alias("Accounting Date"),
    col("exception").alias("Exception"),
)

total_exceptions = exception_output.count()
print(f"[INFO] task 2 exception output rows={total_exceptions}")
exception_output.groupBy("Exception").count().orderBy(col("count").desc()).show(truncate=False)


[INFO] task 2 exception output rows=788


+-----------------+-----+
|Exception        |count|
+-----------------+-----+
|GL_MISMATCH      |403  |
|MAPPING_NOT_FOUND|385  |
+-----------------+-----+



In [15]:
print(f"[PASS] task 2 accounting mapping validation: exception_rows={total_exceptions}")
spark.stop()


[PASS] task 2 accounting mapping validation: exception_rows=788


## Task 3 - Investigate a Finance Variance

Finance reports a material variance between the expected and platform closing balances. The dataset
may contain any combination of duplicate accounting entries, transactions posted twice, incorrect
debit/credit indicators, incorrect FX conversion, missing accounting mappings, transactions posted
one accounting day late, incorrect legal-entity allocation, and incorrect cost-center assignment.
Each is checked independently below and the total variance is decomposed across whichever categories
this run's data actually shows.


In [16]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task3-variance-investigation")
    .getOrCreate()
)

txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")
gl_df = jdbc_table("finance.gl_balance")
txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")


### Duplicate accounting entries and transactions posted twice

Hash the business fields (every column except `transaction_id`) and find hash collisions across
distinct transaction ids posted on the same posting date - every row in a matching group is a
candidate duplicate. This single detection surfaces both assignment-named scenarios equally: a
straight duplicate entry and a transaction re-posted under a different id are indistinguishable from
the data alone, since both manifest as more than one transaction id carrying identical business-field
content on the same posting date.


In [17]:
dup_check = spark.sql('''
    SELECT transaction_id, account_id, posting_date, local_amount,
           MD5(CONCAT_WS('|', account_id, posting_date, transaction_amount, currency,
                          debit_credit_indicator, product_code, gl_account, cost_center)) AS entry_hash
    FROM finance_transactions
''')

from pyspark.sql import Window
from pyspark.sql.functions import row_number, count as spark_count

dup_window = Window.partitionBy("entry_hash", "posting_date").orderBy("transaction_id")
dup_ranked = dup_check.withColumn("rn", row_number().over(dup_window))
dup_group_size = dup_check.groupBy("entry_hash", "posting_date").agg(spark_count("*").alias("group_size"))

duplicate_entries = (
    dup_ranked.join(dup_group_size, ["entry_hash", "posting_date"])
    .filter((col("group_size") > 1) & (col("rn") > 1))
)

duplicate_rows = duplicate_entries.count()
duplicate_amount = duplicate_entries.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
print(f"[INFO] duplicate/re-posted accounting entries: rows={duplicate_rows} amount={duplicate_amount:.2f}")
duplicate_entries.select("transaction_id", "account_id", "posting_date", "local_amount").show(10, truncate=False)


[INFO] duplicate/re-posted accounting entries: rows=21 amount=255845.35


+--------------+-----------+------------+------------+
|transaction_id|account_id |posting_date|local_amount|
+--------------+-----------+------------+------------+
|FTX-DUP001503 |ACC-1000101|2026-08-20  |12574.31    |
|FTX-DUP001519 |ACC-1000336|2026-08-20  |16248.67    |
|FTX-DUP001511 |ACC-1000130|2026-08-17  |3457.66     |
|FTX-DUP001510 |ACC-1000282|2026-08-18  |17099.48    |
|FTX-DUP001517 |ACC-1000378|2026-08-19  |2550.32     |
|FTX-DUP001512 |ACC-1000497|2026-08-20  |16560.67    |
|FTX-DUP001514 |ACC-1000411|2026-08-20  |5873.85     |
|FTX-DUP001509 |ACC-1000441|2026-08-18  |1859.77     |
|FTX-DUP001521 |ACC-1000158|2026-08-18  |5577.10     |
|FTX-DUP001504 |ACC-1000069|2026-08-19  |12533.18    |
+--------------+-----------+------------+------------+
only showing top 10 rows



### Incorrect debit/credit indicator

Without a dedicated GL normal-balance reference, a flipped indicator is detected indirectly through
the movement recomputation: flipping DEBIT/CREDIT sends a transaction's amount to the wrong side of
the sum, producing an equal-and-opposite debit/credit variance at the same key rather than a one-sided
miss - candidates are keys where the two variances cancel but neither is individually zero.


In [18]:
txn = txn_df.select(
    col("posting_date").alias("accounting_date"),
    col("legal_entity"), col("gl_account"), col("cost_center"), col("currency"),
    col("debit_credit_indicator"),
    col("local_amount").cast("double").alias("local_amount"),
)
gl = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"), col("cost_center"), col("currency"),
    col("debit_movement").cast("double").alias("debit_movement"),
    col("credit_movement").cast("double").alias("credit_movement"),
)
recomputed = txn.groupBy("accounting_date", "legal_entity", "gl_account", "cost_center", "currency").agg(
    spark_sum(when(col("debit_credit_indicator") == "DEBIT", col("local_amount")).otherwise(0.0)).alias("recomputed_debit"),
    spark_sum(when(col("debit_credit_indicator") == "CREDIT", col("local_amount")).otherwise(0.0)).alias("recomputed_credit"),
)
JOIN_KEYS = ["accounting_date", "legal_entity", "gl_account", "cost_center", "currency"]
movement_compare = gl.join(recomputed, JOIN_KEYS, "full_outer").fillna(
    0.0, subset=["debit_movement", "recomputed_debit", "credit_movement", "recomputed_credit"]
).withColumn("debit_variance", col("debit_movement") - col("recomputed_debit")
).withColumn("credit_variance", col("credit_movement") - col("recomputed_credit"))

wrong_indicator_keys = movement_compare.filter(
    (spark_abs(col("debit_variance") + col("credit_variance")) < 0.01)
    & (spark_abs(col("debit_variance")) > 0.01)
)
wrong_indicator_count = wrong_indicator_keys.count()
print(f"[INFO] incorrect debit/credit indicator candidate keys={wrong_indicator_count}")
wrong_indicator_keys.select(*JOIN_KEYS, "debit_variance", "credit_variance").show(10, truncate=False)


[INFO] incorrect debit/credit indicator candidate keys=0


+---------------+------------+----------+-----------+--------+--------------+---------------+
|accounting_date|legal_entity|gl_account|cost_center|currency|debit_variance|credit_variance|
+---------------+------------+----------+-----------+--------+--------------+---------------+
+---------------+------------+----------+-----------+--------+--------------+---------------+



### Incorrect FX conversion

`local_amount` is expected to equal `transaction_amount * exchange_rate`, rounded to two decimal
places. Rows more than one minor-currency-unit outside that expectation are flagged.


In [19]:
from pyspark.sql.functions import round as spark_round

fx_check = txn_df.select(
    "transaction_id", "transaction_amount", "exchange_rate",
    col("local_amount").cast("double").alias("local_amount"),
).withColumn(
    "expected_local_amount", spark_round(col("transaction_amount") * col("exchange_rate"), 2)
).withColumn(
    "fx_variance", col("local_amount") - col("expected_local_amount")
)

fx_mismatches = fx_check.filter(spark_abs(col("fx_variance")) > 0.01)
fx_mismatch_rows = fx_mismatches.count()
fx_mismatch_amount = fx_mismatches.agg(spark_sum("fx_variance")).collect()[0][0] or 0.0
print(f"[INFO] incorrect FX conversion rows={fx_mismatch_rows} amount={fx_mismatch_amount:.2f}")
fx_mismatches.show(10, truncate=False)


[INFO] incorrect FX conversion rows=3 amount=4467.53


+--------------+------------------+-------------+------------+---------------------+------------------+
|transaction_id|transaction_amount|exchange_rate|local_amount|expected_local_amount|fx_variance       |
+--------------+------------------+-------------+------------+---------------------+------------------+
|FTX-0000072   |18900.56          |1.32154324   |27475.7     |24977.91             |2497.790000000001 |
|FTX-0001199   |529.34            |1.34399564   |782.57      |711.43               |71.1400000000001  |
|FTX-0001350   |14159.60          |1.34085776   |20884.61    |18986.01             |1898.6000000000022|
+--------------+------------------+-------------+------------+---------------------+------------------+



### Missing accounting mapping's variance contribution

A transaction with no valid accounting mapping cannot be confirmed against an expected GL account, so
its value is reported here as unmapped variance rather than folded into the GL-mismatch count from
Task 2 - the sum of `local_amount` across every transaction with no accounting mapping at all or none
covering its transaction date.


In [20]:
unmapped = spark.sql('''
    SELECT t.transaction_id, t.local_amount
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    )
''')

unmapped_rows = unmapped.count()
unmapped_amount = unmapped.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
print(f"[INFO] missing accounting mapping variance contribution: rows={unmapped_rows} amount={unmapped_amount:.2f}")


[INFO] missing accounting mapping variance contribution: rows=385 amount=4387367.89


### Transaction posted one accounting day late

Candidates are transactions whose posting date is exactly one calendar day after the transaction
date. Each candidate is cross-checked against the per-day movement variance from Task 1 before being
counted - flagged only if moving it to the prior day materially improves that prior day's debit or
credit variance (its own amount closely matches the prior day's shortfall), so a transaction that
legitimately posts a day later (e.g. a weekend transaction posted the next business day) is not
over-flagged.


In [21]:
from pyspark.sql.functions import date_add

late_candidates = txn_df.select(
    "transaction_id", "posting_date", "transaction_date", "legal_entity", "gl_account",
    "cost_center", "currency", "debit_credit_indicator",
    col("local_amount").cast("double").alias("local_amount"),
).filter(col("posting_date") == date_add(col("transaction_date"), 1))

late_candidate_count = late_candidates.count()
print(f"[INFO] posted-one-day-late candidates={late_candidate_count}")

# cross-check: does moving the candidate to transaction_date materially improve that prior day's
# debit/credit variance at the same five-key grain?
prior_day_variance = movement_compare.select(
    col("accounting_date").alias("prior_date"), "legal_entity", "gl_account", "cost_center", "currency",
    "debit_variance", "credit_variance",
)
late_checked = late_candidates.join(
    prior_day_variance,
    (late_candidates.transaction_date == prior_day_variance.prior_date)
    & (late_candidates.legal_entity == prior_day_variance.legal_entity)
    & (late_candidates.gl_account == prior_day_variance.gl_account)
    & (late_candidates.cost_center == prior_day_variance.cost_center)
    & (late_candidates.currency == prior_day_variance.currency),
    "left",
).withColumn(
    "relevant_variance", when(col("debit_credit_indicator") == "DEBIT", col("debit_variance")).otherwise(col("credit_variance"))
)

late_confirmed = late_checked.filter(
    spark_abs(spark_abs(col("relevant_variance")) - col("local_amount")) < 1.0
)
late_confirmed_count = late_confirmed.count()
late_confirmed_amount = late_confirmed.agg(spark_sum("local_amount")).collect()[0][0] or 0.0
print(f"[INFO] posted-one-day-late confirmed against prior-day shortfall: rows={late_confirmed_count} amount={late_confirmed_amount:.2f}")


[INFO] posted-one-day-late candidates=12


[INFO] posted-one-day-late confirmed against prior-day shortfall: rows=0 amount=0.00


### Incorrect legal-entity allocation

The mapping table carries no expected-legal-entity column, so this is a majority-vote check per
account: an account's legal entity is expected to be stable, so a transaction whose legal entity
disagrees with that account's most-frequent posted value elsewhere in the period is a probable
misallocation.


In [22]:
account_entity_mode = spark.sql('''
    SELECT account_id, legal_entity, COUNT(*) AS n,
           ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY COUNT(*) DESC) AS rnk
    FROM finance_transactions
    GROUP BY account_id, legal_entity
''').filter(col("rnk") == 1)

wrong_entity = txn_df.alias("t").join(
    account_entity_mode.alias("e"), col("t.account_id") == col("e.account_id")
).filter(col("t.legal_entity") != col("e.legal_entity")).select(
    col("t.transaction_id"), col("t.legal_entity").alias("actual_entity"), col("e.legal_entity").alias("expected_entity"),
    col("t.local_amount").cast("double").alias("local_amount"),
)

wrong_entity_rows = wrong_entity.count()
wrong_entity_amount = wrong_entity.agg(spark_sum("local_amount")).collect()[0][0] or 0.0
print(f"[INFO] incorrect legal-entity allocation: rows={wrong_entity_rows} amount={wrong_entity_amount:.2f}")
wrong_entity.show(10, truncate=False)


[INFO] incorrect legal-entity allocation: rows=6 amount=60697.50


+--------------+-------------+---------------+------------+
|transaction_id|actual_entity|expected_entity|local_amount|
+--------------+-------------+---------------+------------+
|FTX-0001392   |LE1          |LE3            |18465.48    |
|FTX-0001387   |LE1          |LE2            |1323.82     |
|FTX-0000775   |LE1          |LE2            |15558.97    |
|FTX-0000896   |LE3          |LE4            |12251.32    |
|FTX-0000706   |LE3          |LE1            |120.03      |
|FTX-0000519   |LE2          |LE4            |12977.88    |
+--------------+-------------+---------------+------------+



### Incorrect cost-center assignment

Unlike legal entity, the mapping table carries an expected cost center, so this reuses the same
effective-dated join as the accounting mapping validation.


In [23]:
wrong_cost_center = spark.sql('''
    SELECT t.transaction_id, t.cost_center AS actual_cost_center, m.expected_cost_center, t.local_amount
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
      AND t.transaction_date >= m.effective_start_date
      AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    WHERE t.cost_center <> m.expected_cost_center
''')

# a transaction matching more than one mapping row (see the overlapping/multi-GL mapping checks in
# task 2) is evaluated against each match independently, so it can appear more than once here -
# report both the raw row count and the distinct-transaction count, as task 2 did for GL_MISMATCH.
wrong_cc_rows = wrong_cost_center.count()
wrong_cc_txns = wrong_cost_center.select("transaction_id").distinct().count()
wrong_cc_amount = (
    wrong_cost_center.dropDuplicates(["transaction_id"])
    .agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
)
print(f"[INFO] incorrect cost-center assignment: rows={wrong_cc_rows} distinct_transactions={wrong_cc_txns} amount={wrong_cc_amount:.2f}")
wrong_cost_center.show(10, truncate=False)


[INFO] incorrect cost-center assignment: rows=11 distinct_transactions=8 amount=85485.13


+--------------+------------------+--------------------+------------+
|transaction_id|actual_cost_center|expected_cost_center|local_amount|
+--------------+------------------+--------------------+------------+
|FTX-0000080   |CC04              |CC08                |5773.16     |
|FTX-0000080   |CC04              |CC08                |5773.16     |
|FTX-0001297   |CC08              |CC05                |11453.83    |
|FTX-0001297   |CC08              |CC05                |11453.83    |
|FTX-0001297   |CC08              |CC05                |11453.83    |
|FTX-0001452   |CC02              |CC07                |1355.23     |
|FTX-0000660   |CC03              |CC04                |17900.46    |
|FTX-0000068   |CC09              |CC02                |16506.33    |
|FTX-0001358   |CC99              |CC02                |9706.73     |
|FTX-0001341   |CC02              |CC01                |22174.44    |
+--------------+------------------+--------------------+------------+
only showing top 10 

### Variance decomposition

Each category's contribution to the total value in the transaction data not otherwise reconciled
against the Ledger, summed and compared bottom-up against the single top-line variance between the
Ledger's total and the independently recomputed transaction total from Task 1 - the same top-down
figure the scenario's own reported variance is an instance of.


In [24]:
decomposition = [
    ("duplicate / re-posted accounting entries", duplicate_rows, duplicate_amount),
    ("incorrect debit/credit indicator (candidate keys)", wrong_indicator_count, None),
    ("incorrect FX conversion", fx_mismatch_rows, fx_mismatch_amount),
    ("missing accounting mapping", unmapped_rows, unmapped_amount),
    ("posted one accounting day late (confirmed)", late_confirmed_count, late_confirmed_amount),
    ("incorrect legal-entity allocation", wrong_entity_rows, wrong_entity_amount),
    ("incorrect cost-center assignment", wrong_cc_txns, wrong_cc_amount),
]

print("[INFO] category decomposition:")
for name, rows, amount in decomposition:
    amount_str = f"{amount:.2f}" if amount is not None else "n/a (indicator flip nets to zero)"
    print(f"  {name}: rows={rows} amount={amount_str}")

# top-down: gl_amount and txn_amount are the same totals task 1 already computed
# (SUM(closing_balance) from the Ledger, SUM(local_amount) from the transactions) - reused here
# rather than recomputed, so there is one source of truth for this figure.
independent_gl_variance = gl_amount - txn_amount

# bottom-up: sum of the magnitudes attributed above (categories with no confirmed value contribute 0)
bottom_up_sum = sum(amount for _, _, amount in decomposition if amount is not None)

unexplained_residual = abs(independent_gl_variance) - bottom_up_sum

print(f"[INFO] independent Ledger total variance (Ledger total - transaction total): {independent_gl_variance:.2f}")
print(f"[INFO] bottom-up sum of category contributions: {bottom_up_sum:.2f}")
print(f"[INFO] unexplained residual (|independent variance| - bottom-up sum): {unexplained_residual:.2f}")


[INFO] category decomposition:
  duplicate / re-posted accounting entries: rows=21 amount=255845.35
  incorrect debit/credit indicator (candidate keys): rows=0 amount=n/a (indicator flip nets to zero)
  incorrect FX conversion: rows=3 amount=4467.53
  missing accounting mapping: rows=385 amount=4387367.89
  posted one accounting day late (confirmed): rows=0 amount=0.00
  incorrect legal-entity allocation: rows=6 amount=60697.50
  incorrect cost-center assignment: rows=8 amount=85485.13
[INFO] independent Ledger total variance (Ledger total - transaction total): -4002303.12
[INFO] bottom-up sum of category contributions: 4793863.40
[INFO] unexplained residual (|independent variance| - bottom-up sum): -791560.28


### Testing the residual against two predicted, quantifiable causes

The residual above should not be assumed unexplained without checking two specific, testable
predictions first:

1. **dimensional-only categories** - incorrect legal-entity allocation and incorrect cost-center
   assignment reassign a transaction between sub-totals without changing the Ledger's grand total, so
   their value should not have been counted toward the bottom-up sum at all.
2. **overlap between the remaining categories** - duplicate/re-posted entries, incorrect FX
   conversion, and missing accounting mapping are each detected by an independent check; a single
   transaction flagged by more than one of them has its value summed once per check, double-counting
   it relative to the transaction total the top-down figure is measured against.

Both are computed directly below, not assumed.


In [25]:
dimensional_only_amount = wrong_entity_amount + wrong_cc_amount

overlap_check = (
    duplicate_entries.select("transaction_id", col("local_amount").cast("double").alias("local_amount"))
    .unionByName(fx_mismatches.select("transaction_id", col("local_amount").cast("double").alias("local_amount")))
    .unionByName(unmapped.select("transaction_id", col("local_amount").cast("double").alias("local_amount")))
)
gross_sum_with_repeats = overlap_check.agg(spark_sum("local_amount")).collect()[0][0] or 0.0
distinct_sum = overlap_check.dropDuplicates(["transaction_id"]).agg(spark_sum("local_amount")).collect()[0][0] or 0.0
double_counted_excess = gross_sum_with_repeats - distinct_sum

adjusted_bottom_up_sum = bottom_up_sum - dimensional_only_amount - double_counted_excess
adjusted_residual = abs(independent_gl_variance) - adjusted_bottom_up_sum

explained_fraction = 1 - abs(adjusted_residual) / abs(unexplained_residual) if unexplained_residual else 0.0

print(f"[INFO] dimensional-only categories removed from the bottom-up sum: {dimensional_only_amount:.2f}")
print(f"[INFO] value double-counted by more than one of the remaining checks: {double_counted_excess:.2f}")
print(f"[INFO] adjusted bottom-up sum: {adjusted_bottom_up_sum:.2f}")
print(f"[INFO] adjusted residual: {adjusted_residual:.2f}")
print(f"[INFO] share of the original residual these two predictions explain: {explained_fraction*100:.1f}%")


[INFO] dimensional-only categories removed from the bottom-up sum: 146182.63
[INFO] value double-counted by more than one of the remaining checks: 125566.69
[INFO] adjusted bottom-up sum: 4522114.08
[INFO] adjusted residual: -519810.96
[INFO] share of the original residual these two predictions explain: 34.3%


In [26]:
print("[PASS] task 3 variance investigation: all eight categories checked")
spark.stop()


[PASS] task 3 variance investigation: all eight categories checked


## Exception Dataset

One row per flagged transaction/issue-type pair, minimum columns: transaction id, issue type, source
value, comparison value, variance. Combines the accounting-mapping-validation findings (Task 2) and
the variance-investigation findings (Task 3) into a single dataset.


In [27]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-exception-dataset")
    .getOrCreate()
)

txn_df = jdbc_table("bronze.finance_transactions")
map_df = jdbc_table("ref.accounting_mapping")
gl_df = jdbc_table("finance.gl_balance")
txn_df.createOrReplaceTempView("finance_transactions")
map_df.createOrReplaceTempView("accounting_mapping")

mapping_exceptions_ds = spark.sql('''
    SELECT t.transaction_id, 'GL_MISMATCH' AS issue_type,
           t.gl_account AS source_value, m.expected_gl_account AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON  t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
      AND t.transaction_date >= m.effective_start_date
      AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    WHERE t.gl_account <> m.expected_gl_account

    UNION ALL

    SELECT t.transaction_id, 'NO_EFFECTIVE_MAPPING' AS issue_type,
           t.gl_account AS source_value, CAST(NULL AS STRING) AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    WHERE EXISTS (
      SELECT 1 FROM accounting_mapping m WHERE m.product_code = t.product_code
        AND m.transaction_type = t.debit_credit_indicator)
    AND NOT EXISTS (
      SELECT 1 FROM accounting_mapping m WHERE m.product_code = t.product_code
        AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL))

    UNION ALL

    SELECT t.transaction_id, 'MAPPING_NOT_FOUND' AS issue_type,
           t.gl_account AS source_value, CAST(NULL AS STRING) AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m WHERE m.product_code = t.product_code
        AND m.transaction_type = t.debit_credit_indicator)

    UNION ALL

    SELECT t.transaction_id, 'EXPIRED_MAPPING' AS issue_type,
           t.gl_account AS source_value, m.expected_gl_account AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
    WHERE m.effective_end_date IS NOT NULL
      AND t.transaction_date > m.effective_end_date
      AND NOT EXISTS (
        SELECT 1 FROM accounting_mapping m2
        WHERE m2.product_code = t.product_code AND m2.transaction_type = t.debit_credit_indicator
          AND t.transaction_date >= m2.effective_start_date
          AND (t.transaction_date <= m2.effective_end_date OR m2.effective_end_date IS NULL))

    UNION ALL

    SELECT t.transaction_id, 'WRONG_COST_CENTER' AS issue_type,
           t.cost_center AS source_value, m.expected_cost_center AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN accounting_mapping m
      ON t.product_code = m.product_code AND t.debit_credit_indicator = m.transaction_type
      AND t.transaction_date >= m.effective_start_date
      AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL)
    WHERE t.cost_center <> m.expected_cost_center

    UNION ALL

    SELECT t.transaction_id, 'UNMAPPED_VARIANCE' AS issue_type,
           CAST(t.local_amount AS STRING) AS source_value, CAST(NULL AS STRING) AS comparison_value,
           t.local_amount AS variance
    FROM finance_transactions t
    WHERE NOT EXISTS (
      SELECT 1 FROM accounting_mapping m
      WHERE m.product_code = t.product_code AND m.transaction_type = t.debit_credit_indicator
        AND t.transaction_date >= m.effective_start_date
        AND (t.transaction_date <= m.effective_end_date OR m.effective_end_date IS NULL))
''')

fx_exceptions_ds = spark.sql('''
    SELECT transaction_id, 'FX_CONVERSION_ERROR' AS issue_type,
           CAST(local_amount AS STRING) AS source_value,
           CAST(ROUND(transaction_amount * exchange_rate, 2) AS STRING) AS comparison_value,
           local_amount - ROUND(transaction_amount * exchange_rate, 2) AS variance
    FROM finance_transactions
    WHERE ABS(local_amount - ROUND(transaction_amount * exchange_rate, 2)) > 0.01
''')

duplicate_exceptions_ds = spark.sql('''
    WITH hashed AS (
      SELECT transaction_id, account_id, posting_date, local_amount,
             MD5(CONCAT_WS('|', account_id, posting_date, transaction_amount, currency,
                            debit_credit_indicator, product_code, gl_account, cost_center)) AS entry_hash
      FROM finance_transactions
    ),
    ranked AS (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY entry_hash, posting_date ORDER BY transaction_id) AS rn,
             COUNT(*) OVER (PARTITION BY entry_hash, posting_date) AS group_size
      FROM hashed
    )
    SELECT transaction_id, 'DUPLICATE_ENTRY' AS issue_type,
           CAST(local_amount AS STRING) AS source_value, CAST(NULL AS STRING) AS comparison_value,
           local_amount AS variance
    FROM ranked WHERE group_size > 1 AND rn > 1
''')

wrong_entity_exceptions_ds = spark.sql('''
    WITH account_entity_mode AS (
      SELECT account_id, legal_entity,
             ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY COUNT(*) DESC) AS rnk
      FROM finance_transactions
      GROUP BY account_id, legal_entity
    )
    SELECT t.transaction_id, 'WRONG_LEGAL_ENTITY' AS issue_type,
           t.legal_entity AS source_value, e.legal_entity AS comparison_value,
           CAST(NULL AS DOUBLE) AS variance
    FROM finance_transactions t
    JOIN account_entity_mode e ON t.account_id = e.account_id AND e.rnk = 1
    WHERE t.legal_entity <> e.legal_entity
''')

exception_dataset = (
    mapping_exceptions_ds
    .unionByName(fx_exceptions_ds)
    .unionByName(duplicate_exceptions_ds)
    .unionByName(wrong_entity_exceptions_ds)
)
exception_dataset.cache()
total_exception_rows = exception_dataset.count()
print(f"[INFO] exception dataset rows={total_exception_rows}")
exception_dataset.groupBy("issue_type").count().orderBy(col("count").desc()).show(truncate=False)
exception_dataset.show(10, truncate=False)


[INFO] exception dataset rows=1214


+-------------------+-----+
|issue_type         |count|
+-------------------+-----+
|GL_MISMATCH        |403  |
|MAPPING_NOT_FOUND  |385  |
|UNMAPPED_VARIANCE  |385  |
|DUPLICATE_ENTRY    |21   |
|WRONG_COST_CENTER  |11   |
|WRONG_LEGAL_ENTITY |6    |
|FX_CONVERSION_ERROR|3    |
+-------------------+-----+



+--------------+-----------+------------+----------------+--------+
|transaction_id|issue_type |source_value|comparison_value|variance|
+--------------+-----------+------------+----------------+--------+
|FTX-0000008   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000012   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000020   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000023   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000027   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000042   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000080   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000100   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000114   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
|FTX-0000156   |GL_MISMATCH|GL1007      |GL1014          |NULL    |
+--------------+-----------+------------+----------------+--------+
only showing top 10 rows



In [28]:
print(f"[PASS] exception dataset: rows={total_exception_rows}")
spark.stop()


[PASS] exception dataset: rows=1214
